# Data Cleaning

Import libraries/packages + raw data

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
import numpy as np

DATA_DIR = BASE_PATH / "data" / "raw"
# Import Data
df = pd.read_excel(DATA_DIR / "taiwan_ORN_raw.xlsx")
df = df.drop(["ORN"], axis=1)
df = df.rename(columns={"ORN ": "ORN"})
df = df.dropna(subset=["ORN"])

Import, subset, and rename columns --> only include patients with Radiation Therapy

In [ ]:
rename_dict = {
    ## Pre-Op chars
    "AGE": "AGE",
    "BMI": "BMI",
    "GENDER (0=female, 1=male)": "SEX",
    "DM(0=no, 1=yes)": "Diabetes",
    "ASA, 0= ASA I or II, 1=ASA III": "ASA",
    "PRIOREX( prior opearion at same side, 0=no, 1=yes)": "PRIOREX",
    "PRECT(pre-op chemotherapy)": "PRECT",
    "PRERT( pre-op RT)": "PRERT",
    ## Disease Chars
    "RECUR(0=primary, 1=recurrence)": "RECUR",
    "SITE(1= mouth floor,  2=buccal, 3=retromolar, 4=gum, 5=tongue , 6=lip)": "SITE",
    "T(0=T1or2, 1=T3or4)": "SIZE",
    "N(0= (-), 1=(+))": "LYMPH",
    "OVERALLSTAGE(1= stage1, 2=stage2/3, 3=stage4)": "STAGE",
    "DEFECTTYPE(0= intraoral only, 1=composite defect)": "DEFECT",
    "SECONDPRIMARY(0=no, 1=yes)": "SECONDPRIMARY",
    ## Surg Chars
    "LENGTH(defect length)": "LENGTH",
    "JEWER(Jewer's classification, 0=C, 1=L, 3=LC, 5=LCL)": "JEWER",
    "OSTEOTOMY(no. of osteostomy of the fibula bone)": "OSTEOTOMY",
    "PLATE(0=mini plate 1=reconstruction plate 2=preformed plate)": "PLATE",
    "FLAP(0=OSC flap, 1=chimeric flap with muscle)": "FLAP",
    "BT(intra-op blood transfusion, 0=no, 1=yes)": "TRANSFUS",
    "ISCHEMICTIME": "ISCHEMICTIME",
    "OPTIME": "OPTIME",
    ## Immediate post-Op Chars
    "REOPEN": "REOP",
    "ADMISSION( hospitalization days)": "LOHS",
    "POSTRT(post-op RT)": "POSTRT",
    "POSTCT(post-op chemotherapy)": "POSTCT",
    "WOUNDINF(post-op wound infection)": "WOUNDINF",
    "HGB(post-op hemoglobin)": "HGB",
    "ALB(post-op albumin)": "ALB",
    ## Long term post-op
    "EXPOSURE( plate exposure)": "EXPOSURE",
    "MEDEXPOSURE( treat plate exposure with medication only)": "MEDUSED",
    "TXEXPOSURE(treatments other than medication, 1=plate removal, 2=another flap)": "SURGUSED",
    "EXPOSUREFU( time from op to plate exposure)": "PLATETIME",
    "FU": "FOLLOWTIME",
    ## Target
    "ORN": "ORN",
}
df_rename = df.rename(columns=rename_dict)
df_sub = df_rename[rename_dict.values()].copy()
df_sub.shape
# Subset only RT patients
df_sub = df_sub[(df_sub["POSTRT"] == 1) | (df_sub["PRERT"] == 1)]
# Combine exposure 2 (1 instance) --> exposure 1
df_sub["EXPOSURE"] = df_sub["EXPOSURE"].replace({2: 1})
df_sub.shape

Deal with NAs

- Ischemictime --> 2 999s--> NA
- ALB --> 8 999s --> NA

In [ ]:
df_sub["ISCHEMICTIME"] = np.where(
    df_sub["ISCHEMICTIME"] == 999, np.nan, df_sub["ISCHEMICTIME"]
)
df_sub["ALB"] = np.where(df_sub["ALB"] == 999, np.nan, df_sub["ALB"])

In [ ]:
for col in df_sub:
    val_counts = df_sub[col].value_counts()
    len_counts = len(val_counts)
    n_na = df_sub[col].isna().sum()
    if len_counts > 10:
        n_99 = (df_sub[col] > 900).sum()
        n_neg = (df_sub[col] < 0).sum()
        if n_99 > 0 or n_neg > 0 or n_na > 0:
            print(col)
            print(f"Num 999s: {n_99}")
            print(f"Num negs: {n_neg}")
            print(f"Num NAs: {n_na}")
    else:
        if n_na > 0:
            print(col)
            print(f"Num NAs: {n_na}")

Reformat exposure vars

Time messed up bc of excel formatting

In [ ]:
df_clean = df_sub.copy()
## Time to plate exposure
df_clean["PLATETIME"] = pd.cut(
    df_clean["PLATETIME"],
    bins=[0, 1, 2, float("inf")],
    labels=["0to1", "1to2", "2plus"],
    right=False,  # means [0, 1) is "Early", [1, 2) is "Intermediate", [2, inf) is "Late"
)
df_clean["PLATETIME"] = np.where(
    df_clean["EXPOSURE"] == 0, "NoExposure", df_clean["PLATETIME"]
)

## Medication Used for plate
df_clean["MEDUSED"] = np.select(
    [
        df_clean["EXPOSURE"] == 0,
        (df_clean["EXPOSURE"] == 1) & (df_clean["MEDUSED"] == 1),  # Exp + med used
        (df_clean["EXPOSURE"] == 1) & (df_clean["MEDUSED"] == 0),  # Exp + no med used
    ],
    ["NoExposure", "Yes", "No"],
    default="Unknown",
)
## Surgical Intervention Used for plate
df_clean["SURGUSED"] = np.select(
    [
        df_clean["EXPOSURE"] == 0,
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 0),  # None
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 1),  # plate
        (df_clean["EXPOSURE"] == 1) & (df_clean["SURGUSED"] == 2),  # flap
    ],
    ["NoExposure", "None", "Plate", "Flap"],
    default="Unknown",
)
df_clean["SURGUSED"].value_counts()

Simplify features

In [ ]:
# Change two '3' entries in OSTEOTOMY to '2' for simplicity
df_clean["OSTEOTOMY"] = df_clean["OSTEOTOMY"].replace({3.0: 2.0})
# Rename ASA entries to make binary nature more clear (2-->0, 3-->1)
df_clean["ASA"] = df_clean["ASA"].replace({2: 0, 3: 1})
# Combine 2 'JEWER 0' (C) with 71 'JEWER 5' (LCL) b/c of low frequency
df_clean["JEWER"] = df_clean["JEWER"].replace({0.0: 5.0})
# Combine pre/post RT
## Surgical Intervention Used for plate
df_clean["RADTIME"] = np.select(  # note that no instances of neither
    [
        (df_clean["POSTRT"] == 1) & (df_clean["PRERT"] == 0),  # Just post
        (df_clean["POSTRT"] == 0) & (df_clean["PRERT"] == 1),  # Just pre
        (df_clean["POSTRT"] == 1) & (df_clean["PRERT"] == 1),  # Both
    ],
    ["Post", "Pre", "Both"],
    default="Unknown",
)
df_clean = df_clean.drop(["PRERT", "POSTRT"], axis=1)
df_clean["RADTIME"].value_counts()

Export

In [ ]:
save_path = DATA_DIR / "Cleaned_ORN.parquet"
if save_path.exists():
    save_path.unlink()
df_clean.to_parquet(save_path)